In [0]:
data = [("Raj", "Mumbai", 75000, "Engineering"),
        ("Priya", "Delhi", 95000, "Marketing"),
        ("Amit", "Pune", 45000, "Engineering")]

df = spark.createDataFrame(data, ["name", "city", "salary", "department"])
df.write.format("delta").mode("overwrite").saveAsTable("delta_table")


from delta.tables import DeltaTable
deltatb = DeltaTable.forName(spark, "delta_table")

In [0]:
deltatb.history().select("version","operation","timestamp").show()

+-------+--------------------+-------------------+
|version|           operation|          timestamp|
+-------+--------------------+-------------------+
|      0|CREATE OR REPLACE...|2026-04-24 06:13:21|
+-------+--------------------+-------------------+



In [0]:
# check history 
spark.table("delta_table").show()


deltatb.update("city = 'Mumbai'",{"salary" :"salary * 1.10 "})
deltatb.delete("department = 'Marketing'")
deltatb.history().select("version","operation","timestamp").show()

spark.table('delta_table').show()

+-----+------+------+-----------+
| name|  city|salary| department|
+-----+------+------+-----------+
|  Raj|Mumbai|109807|Engineering|
|Priya| Delhi| 95000|  Marketing|
| Amit|  Pune| 45000|Engineering|
+-----+------+------+-----------+

+-------+--------------------+--------------------+
|version|           operation|           timestamp|
+-------+--------------------+--------------------+
|     15|              DELETE| 2026-04-24 06:30:25|
|     14|            OPTIMIZE| 2026-04-24 06:30:23|
|     13|              UPDATE| 2026-04-24 06:30:20|
|     12|            OPTIMIZE| 2026-04-24 06:29:55|
|     11|              DELETE| 2026-04-24 06:29:54|
|     10|              UPDATE| 2026-04-24 06:29:53|
|      9|            OPTIMIZE| 2026-04-24 06:29:20|
|      8|              DELETE| 2026-04-24 06:29:19|
|      7|              UPDATE| 2026-04-24 06:29:17|
|      6|            OPTIMIZE|2026-04-24 06:28:...|
|      5|              DELETE| 2026-04-24 06:28:57|
|      4|              UPDATE| 20

In [0]:
updates = spark.createDataFrame([
    ("Raj", "Bangalore", 90000, "Engineering"),  # existing data with updated information 
    ("Sara", "Mumbai", 70000, "Finance")          # new data — insert
], ["name", "city", "salary", "department"])

deltatb.alias("t").merge(updates.alias("s"), "t.name = s.name").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
spark.table("delta_table").show()

+----+---------+------+-----------+
|name|     city|salary| department|
+----+---------+------+-----------+
|Amit|     Pune| 45000|Engineering|
| Raj|Bangalore| 90000|Engineering|
|Sara|   Mumbai| 70000|    Finance|
+----+---------+------+-----------+



In [0]:
# TimeTravel 
# keep in mind version 0 - is alwasy original version 
spark.read.format("delta").option("versionAsOf",0).table("delta_table").show()

+-----+------+------+-----------+
| name|  city|salary| department|
+-----+------+------+-----------+
|  Raj|Mumbai| 75000|Engineering|
|Priya| Delhi| 95000|  Marketing|
| Amit|  Pune| 45000|Engineering|
+-----+------+------+-----------+



In [0]:
# restoring to version 0
deltatb.restoreToVersion(0)
spark.table("delta_table").show()

+-----+------+------+-----------+
| name|  city|salary| department|
+-----+------+------+-----------+
|  Raj|Mumbai| 75000|Engineering|
|Priya| Delhi| 95000|  Marketing|
| Amit|  Pune| 45000|Engineering|
+-----+------+------+-----------+



In [0]:
#optimize but first check file count 
spark.sql("optimize delta_table zorder by (city)")
deltatb.history().select("version","operation").show()

+-------+--------------------+
|version|           operation|
+-------+--------------------+
|     19|             RESTORE|
|     18|            OPTIMIZE|
|     17|               MERGE|
|     16|            OPTIMIZE|
|     15|              DELETE|
|     14|            OPTIMIZE|
|     13|              UPDATE|
|     12|            OPTIMIZE|
|     11|              DELETE|
|     10|              UPDATE|
|      9|            OPTIMIZE|
|      8|              DELETE|
|      7|              UPDATE|
|      6|            OPTIMIZE|
|      5|              DELETE|
|      4|              UPDATE|
|      3|            OPTIMIZE|
|      2|              DELETE|
|      1|              UPDATE|
|      0|CREATE OR REPLACE...|
+-------+--------------------+

